# שיעור 02 - חקירת מסגרת Microsoft Agent  

מסגרת **Microsoft Agent Framework (MAF)** היא מסגרת מאוחדת לבניית סוכני בינה מלאכותית. היא מספקת ארכיטקטורה נקייה, קומפוזבילית עם ארבעה בלוקים בסיסיים:  

- **לקוח (Client)** – מתחבר לנקודת קצה של מודל בינה מלאכותית ומטפל בתקשורת  
- **סוכן (Agent)** – עוטף את הלקוח עם הוראות והגדרות כלים  
- **כלים (Tools)** – מרחיבים את יכולות הסוכן עם פונקציות מותאמות שהמודל יכול לקרוא להן  
- **סשן (Session)** – שומר היסטוריית שיחה לאינטראקציות רב-סבביות  

בשיעור זה, נבנה **סוכן הזמנת טיולים** שבודק זמינות יעד באמצעות המושגים הללו.  


## התקנה


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## הבנת ארכיטקטורת מסגרת הסוכן

מסגרת הסוכן של מיקרוסופט פועלת במבנה שכבתי:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **לקוח** – `FoundryChatClient` מתחבר לפריסת Azure OpenAI. הוא מטפל באימות, בעיצוב הבקשות ובניתוח התגובות.
2. **סוכן** – נוצר מהלקוח באמצעות `provider.create_agent()`, הסוכן משלב גישה למודל עם הוראות (הנחיית מערכת) וכלים.
3. **כלים** – פונקציות פייתון שמקושרות באמצעות `@tool` שהסוכן יכול לקרוא להן כדי לבצע פעולות או להשיג נתונים.
4. **מפגש** – אובייקט `AgentSession` (נוצר באמצעות `agent.create_session()`) שמאחסן את היסטוריית השיחה, ומאפשר דו-שיח מרובה סבבים שבה הסוכן זוכר הקשר קודם.

בואו נבנה כל שכבה שלב אחרי שלב.


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## הוספת כלים עם העיטור @tool

כלים מאפשרים לסוכנים לנקוט פעולות מעבר ליצירת טקסט. העיטור `@tool` ממיר פונקציית פייתון רגילה למשהו שהסוכן יכול לקרוא לו.

נקודות מרכזיות:
- השתמש ב- `Annotated[type, "description"]` כדי שהמודל יבין כל פרמטר.
- מחרוזת התיעוד הופכת לתיאור הכלי שהמודל רואה.
- `approval_mode="never_require"` פירושו שהכלי פועל אוטומטית ללא אישור המשתמש.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## יצירת סוכן עם כלים

עכשיו אנו משלבים את הלקוח, ההוראות, והכלים לסוכן אחד. ה-`instructions` משמשים כהנחיית מערכת — הם מגדירים את אישיותו והתנהגותו של הסוכן.


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## שיחות מרובות סבבים עם Sessions

`AgentSession` (נוצר דרך `agent.create_session()`) שומר מעקב אחר כל ההודעות בשיחה. על ידי העברת אותה session לכל קריאה של `agent.run()`, לסוכן יש גישה להיסטוריה המלאה של השיחה והוא יכול להתייחס להודעות מוקדמות.

אנו מעבירים `tools=[check_destination_availability]` כך שהסוכן יכול לקרוא לבודק הזמינות שלנו במהלך כל סבב.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## סיכום

בשיעור זה חקרתם את ארבעת העמודים של מסגרת הסוכן של מיקרוסופט:

| מונח | מה שלמדת |
|---------|------------------|
| **לקוח** | `FoundryChatClient` מתחבר ל-Azure OpenAI עם אימות מבוסס אישורים |
| **סוכן** | `provider.create_agent()` מקבץ חיבור מודל עם הוראות ושם |
| **כלים** | הדקורטור `@tool` מאפשר לפונקציות פייתון להיות מופעלות על ידי הסוכן |
| **מפגש** | `agent.create_session()` שומר היסטוריית שיחה בכמה סבבים |

אבני הבניין האלה מורכבות יחד כדי ליצור סוכנים שמסוגלים לנהל שיחות טבעיות, להפעיל פונקציות חיצוניות, ולשמור הקשר — הבסיס לדפוסים מתקדמים יותר של סוכנות בשיעורים הבאים.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
